In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
# Import Libraries
import numpy as np
import pandas as pd

In [3]:
# Load Data
train= pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test= pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

print(train.shape)
print(test.shape)

(1460, 81)
(1459, 80)


In [4]:
#column with highest missing value
print(train.isnull().sum().sort_values(ascending=False).head())

PoolQC         1453
MiscFeature    1406
Alley          1369
Fence          1179
MasVnrType      872
dtype: int64


In [5]:
# Drop Columns
cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']

train = train.drop(cols_to_drop, axis=1)
test = test.drop(cols_to_drop, axis=1)

In [6]:
# Separate Features & Target
X = train.drop('SalePrice', axis=1)
y = train['SalePrice']

# Feature Engineering

In [7]:
#total square footage
X['TotalSF'] = X['TotalBsmtSF'] + X['1stFlrSF'] + X['2ndFlrSF']
test['TotalSF'] = test['TotalBsmtSF'] + test['1stFlrSF'] + test['2ndFlrSF']

#house age
X['HouseAge'] = X['YrSold'] - X['YearBuilt']
test['HouseAge'] = test['YrSold'] - test['YearBuilt']

# Total Bathrooms
X['TotalBathrooms'] = (
    X['FullBath']
    + 0.5 * X['HalfBath']
    + X['BsmtFullBath']
    + 0.5 * X['BsmtHalfBath']
)

test['TotalBathrooms'] = (
    test['FullBath']
    + 0.5 * test['HalfBath']
    + test['BsmtFullBath']
    + 0.5 * test['BsmtHalfBath']
)

# RemodAge
X['RemodAge'] = X['YrSold'] - X['YearRemodAdd']
test['RemodAge'] = test['YrSold'] - test['YearRemodAdd']

# Total Porch Area
X['TotalPorchSF'] = (
    X['WoodDeckSF']
    + X['OpenPorchSF']
    + X['EnclosedPorch']
    + X['3SsnPorch']
    + X['ScreenPorch']
)

test['TotalPorchSF'] = (
    test['WoodDeckSF']
    + test['OpenPorchSF']
    + test['EnclosedPorch']
    + test['3SsnPorch']
    + test['ScreenPorch']
)

# HasGarage
X['HasGarage'] = (X['GarageArea'] > 0).astype(int)
test['HasGarage'] = (test['GarageArea'] > 0).astype(int)

In [8]:
# Missing Value Handling
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Numeric -> Median
X[num_cols] = X[num_cols].fillna(X[num_cols].median())
test[num_cols] = test[num_cols].fillna(X[num_cols].median())

# Categorical -> Missing
X[cat_cols] = X[cat_cols].fillna('Missing')
test[cat_cols] = test[cat_cols].fillna('Missing')

#Verify missing value
print(X.isnull().sum().sum())
print(test.isnull().sum().sum())

0
0


In [9]:
#One-Hot Encode
X = pd.get_dummies(X)
test_encoded = pd.get_dummies(test)

X, test_encoded = X.align(
    test_encoded,
    join='left',
    axis=1,
    fill_value=0
)

print(X.shape)
print(test_encoded.shape)

(1460, 293)
(1459, 293)


In [10]:
# Log Transform Target
y= np.log1p(y)
print(y.head())

0    12.247699
1    12.109016
2    12.317171
3    11.849405
4    12.429220
Name: SalePrice, dtype: float64


In [11]:
from xgboost import XGBRegressor

model = XGBRegressor(
    random_state=42,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    objective='reg:squarederror'
)

In [12]:
#Creating Search Space

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators': randint(300, 1000),
    'max_depth': randint(3, 8),
    'learning_rate': uniform(0.01, 0.19),   # 0.01–0.20
    'subsample': uniform(0.7, 0.3),          # 0.7–1.0
    'colsample_bytree': uniform(0.7, 0.3)    # 0.7–1.0
}

search = RandomizedSearchCV(
    estimator = model,
    param_distributions = param_dist,
    n_iter = 10,
    cv = 5 ,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
    
)

In [13]:
#Start the Search
search.fit(X,y)

#View the Best Parameters
print("Best Parameters:")
print(search.best_params_)

#Best Cross-Validation Score
print("\nBest RMSE:")
print(-search.best_score_)

#Best Model
best_model = search.best_estimator_

Best Parameters:
{'colsample_bytree': np.float64(0.7520093960523315), 'learning_rate': np.float64(0.08430151543891574), 'max_depth': 4, 'n_estimators': 687, 'subsample': np.float64(0.7935133228268232)}

Best RMSE:
0.1222107836162282


In [14]:
# Train on Full Data
best_model.fit(X, y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=np.float64(0.7520093960523315), device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None,
             learning_rate=np.float64(0.08430151543891574), max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=687, n_jobs=None,
             num_parallel_tree=None, ...)

In [15]:
# Predict Test Data
prediction = best_model.predict(test_encoded)


#Convert Back to Real Prices
prediction = np.expm1(prediction)
print(prediction)

print(prediction[:10])

[123433.38  171895.44  181013.9   ... 151701.84  110331.914 202705.6  ]
[123433.38 171895.44 181013.9  206928.47 185733.31 171519.78 182859.86
 168454.78 174375.77 123185.13]


In [16]:
#Submission File
submission=pd.DataFrame({
    'Id':test['Id'],
    'SalePrice': prediction
})

submission.to_csv('submission_10v.csv',index=False)

submission.head()

,Id,SalePrice
0,1461,123433.382812
1,1462,171895.437500
2,1463,181013.906250
3,1464,206928.468750
4,1465,185733.312500
